# Traffic Vehicle Detection and Tracking with Ultralytics YOLO

This notebook uploads a traffic video, detects and tracks vehicles with persistent IDs, draws category-specific colored boxes, saves an annotated video, and exports JSON results.

**A video is required for true tracking.** A single image can only produce detection IDs, not movement tracking IDs.

## 1. Install dependencies

Enable a GPU first: **Runtime → Change runtime type → T4 GPU → Save**.

In [ ]:
!pip install -q -U ultralytics opencv-python

## 2. Upload a traffic video

Choose an `.mp4`, `.avi`, or other video file.

In [ ]:

from google.colab import files

uploaded = files.upload()
video_path = "/content/" + next(iter(uploaded))
print("Video path:", video_path)


## 3. Configure the model

Use `yolo26x.pt` for higher accuracy. For separate `auto`, `scooty`, and `bike` categories, upload a custom model and set `model_path` to `/content/best.pt`.

In [ ]:

from ultralytics import YOLO
import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"

# Official COCO model: car, bus, truck, motorcycle, bicycle
model_path = "yolo26x.pt"

# For a custom model, use:
# model_path = "/content/best.pt"

model = YOLO(model_path)
print("Device:", device)
print("Model:", model_path)


## 4. Track vehicles and create the annotated video

In [ ]:

import cv2
import json
from collections import defaultdict

vehicle_classes = {
    "car", "bus", "truck", "motorcycle", "bicycle",
    "auto", "rickshaw", "bike", "scooty"
}

# OpenCV BGR colors
COLORS = {
    "car": (255, 0, 0),
    "bus": (255, 0, 255),
    "truck": (0, 165, 255),
    "motorcycle": (0, 255, 0),
    "bicycle": (255, 255, 0),
    "auto": (0, 255, 255),
    "rickshaw": (0, 255, 255),
    "bike": (0, 128, 255),
    "scooty": (128, 0, 255),
}

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise ValueError(f"Could not open video: {video_path}")

fps = cap.get(cv2.CAP_PROP_FPS) or 25
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

output_video_path = "/content/tracked_vehicles.mp4"
output_json_path = "/content/tracked_vehicle_results.json"

writer = cv2.VideoWriter(
    output_video_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

seen_ids = set()
id_to_category = {}
category_ids = defaultdict(set)
frame_count = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    frame_count += 1

    results = model.track(
        source=frame,
        persist=True,
        tracker="bytetrack.yaml",
        conf=0.15,
        iou=0.5,
        imgsz=1280,
        device=device,
        verbose=False
    )
    result = results[0]

    if result.boxes is not None and result.boxes.id is not None:
        boxes = result.boxes.xyxy.cpu().tolist()
        class_ids = result.boxes.cls.cpu().tolist()
        track_ids = result.boxes.id.cpu().tolist()
        confidences = result.boxes.conf.cpu().tolist()

        for box, class_id, track_id, confidence in zip(
            boxes, class_ids, track_ids, confidences
        ):
            category = result.names[int(class_id)].lower().strip()

            if category not in vehicle_classes:
                continue

            track_id = int(track_id)
            x1, y1, x2, y2 = map(int, box)
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(width - 1, x2), min(height - 1, y2)

            if x2 <= x1 or y2 <= y1:
                continue

            color = COLORS.get(category, (255, 255, 255))
            seen_ids.add(track_id)
            id_to_category[track_id] = category
            category_ids[category].add(track_id)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            label = f"ID {track_id} | {category} {confidence:.2f}"
            (tw, th), base = cv2.getTextSize(
                label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2
            )

            label_y1 = max(0, y1 - th - base - 6)
            label_y2 = label_y1 + th + base + 6
            label_x2 = min(width - 1, x1 + tw + 8)

            cv2.rectangle(
                frame, (x1, label_y1), (label_x2, label_y2), color, -1
            )
            cv2.putText(
                frame, label, (x1 + 4, label_y2 - 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                (255, 255, 255), 2, cv2.LINE_AA
            )

    summary = [
        f"Frame: {frame_count}/{total_frames}",
        f"Unique tracked: {len(seen_ids)}"
    ]
    summary += [
        f"{name}: {len(category_ids[name])}"
        for name in sorted(category_ids)
    ]

    panel_h = 15 + 28 * len(summary)
    overlay = frame.copy()
    cv2.rectangle(overlay, (8, 8), (300, panel_h), (0, 0, 0), -1)
    frame = cv2.addWeighted(overlay, 0.65, frame, 0.35, 0)

    y = 32
    for line in summary:
        cv2.putText(
            frame, line, (16, y),
            cv2.FONT_HERSHEY_SIMPLEX, 0.65,
            (255, 255, 255), 2, cv2.LINE_AA
        )
        y += 28

    writer.write(frame)

cap.release()
writer.release()

vehicle_counts = {
    name: len(ids) for name, ids in category_ids.items()
}

json_output = {
    "source_video": video_path,
    "total_unique_tracked_vehicles": len(seen_ids),
    "vehicle_counts": vehicle_counts,
    "tracked_vehicles": [
        {"track_id": int(track_id), "category": category}
        for track_id, category in sorted(id_to_category.items())
    ]
}

with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(json_output, f, indent=2, ensure_ascii=False)

print(json.dumps(json_output, indent=2))
print("Saved video:", output_video_path)
print("Saved JSON:", output_json_path)


## 5. Display the annotated video

In [ ]:

from IPython.display import HTML, display
from base64 import b64encode

browser_video_path = "/content/tracked_vehicles_browser.mp4"

!ffmpeg -y -loglevel error -i /content/tracked_vehicles.mp4 \
    -vcodec libx264 -pix_fmt yuv420p /content/tracked_vehicles_browser.mp4

with open(browser_video_path, "rb") as f:
    video_data = b64encode(f.read()).decode()

display(HTML(f'''
<video width="900" controls>
  <source src="data:video/mp4;base64,{video_data}" type="video/mp4">
</video>
'''))


## 6. Download the results

In [ ]:

from google.colab import files

files.download("/content/tracked_vehicles_browser.mp4")
files.download("/content/tracked_vehicle_results.json")
